# Component 2 – EDA of AgDRIFT Dataset A
Aim: check data quality, understand what drives spray drift, and decide model inputs.
Dataset: agdrift_dataset_v3_final.csv (5,000 AgDRIFT simulation runs)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

EDA_DIR = Path.cwd()
DATA = EDA_DIR.parent / "data" / "agdrift_dataset_v3_final.csv"
FIG_DIR = EDA_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

def save(name): plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

ORDER = ["VF-F", "F-M", "M-C", "C-VC"]
INPUTS = ["crosswind_ms", "height_m", "swath_m", "temp_c", "rh_pct"]
TARGETS = ["drift_distance_m", "drift_loss_pct", "on_target_pct", "evaporation_pct"]
TARGET = "drift_distance_m"

df = pd.read_csv(DATA)
print(df.shape)

(5000, 17)


## Stage 1 – Dataset overview
**What:** look at the size, column types and value ranges.
**Why:** a first look shows obvious problems (wrong types, impossible values) before any analysis.

In [4]:
df.head()

,run_id,split,datetime,region,wind_speed_ms,wind_dir_deg,temp_c,rh_pct,flight_heading_deg,crosswind_ms,height_m,droplet_class,swath_m,drift_distance_m,drift_loss_pct,on_target_pct,evaporation_pct
0,S00001,train,8/27/2025 12:00,Badulla,1.48,297,33.0,34,144.1,0.67,2.94,M-C,5.51,11.609,23.57915,76.42085,14.81815
1,S00002,test,1/6/2025 6:00,Ratnapura,0.82,149,22.6,93,49.0,0.81,3.42,VF-F,5.72,30.541,39.59527,60.40473,11.67977
2,S00003,train,2/11/2023 13:00,Kandy,1.81,301,27.3,61,18.0,1.76,2.59,F-M,5.61,23.329,35.23222,64.50504,13.19584
3,S00004,train,3/22/2025 11:00,Kandy,2.57,75,31.4,50,168.8,2.56,2.27,VF-F,6.35,48.200,54.43470,44.09075,40.85497
4,S00005,train,3/5/2024 15:00,Kandy,2.33,90,33.0,35,245.7,0.96,3.20,VF-F,5.18,44.149,51.50602,48.49398,54.04653


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   run_id              5000 non-null   object 
 1   split               5000 non-null   object 
 2   datetime            5000 non-null   object 
 3   region              5000 non-null   object 
 4   wind_speed_ms       5000 non-null   float64
 5   wind_dir_deg        5000 non-null   int64  
 6   temp_c              5000 non-null   float64
 7   rh_pct              5000 non-null   int64  
 8   flight_heading_deg  5000 non-null   float64
 9   crosswind_ms        5000 non-null   float64
 10  height_m            5000 non-null   float64
 11  droplet_class       5000 non-null   object 
 12  swath_m             5000 non-null   float64
 13  drift_distance_m    5000 non-null   float64
 14  drift_loss_pct      5000 non-null   float64
 15  on_target_pct       5000 non-null   float64
 16  evapor

In [6]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
wind_speed_ms,5000.0,2.02,1.07,0.52,1.16,1.79,2.68,6.83
wind_dir_deg,5000.0,189.44,110.11,1.00,90.00,231.00,282.00,360.00
temp_c,5000.0,24.73,5.16,7.70,20.80,25.60,28.60,35.90
rh_pct,5000.0,69.74,16.23,19.00,59.00,71.00,82.00,100.00
flight_heading_deg,5000.0,179.23,104.30,0.10,87.45,182.40,266.23,360.00
crosswind_ms,5000.0,1.49,0.90,0.50,0.79,1.22,1.92,6.68
height_m,5000.0,2.75,0.44,2.00,2.37,2.75,3.13,3.50
swath_m,5000.0,5.60,0.46,4.80,5.20,5.59,5.99,6.40
drift_distance_m,5000.0,21.74,11.62,6.79,13.17,17.42,28.42,74.40
drift_loss_pct,5000.0,32.36,15.05,6.00,20.97,28.67,41.48,91.89


Drift distance ranges from 6.8 to 74.4 m; its mean (21.7 m) is above its median (17.4 m), meaning right skew.